# Lähde-ekstraktio: {Author Year}

DOI: `{doi}`  
PDF: `data/raw/{path/to/pdf}`

Tämä notebook tekee viisi vaihetta:

1. **Lukee PDF:n** (avaa Claude Code -sivupaneelista)
2. **LLM-raakaekstraktio** Pydantic-skeemalla `RawPair`
3. **Rikastaa** SMILES:t PubChem:istä (RDKit kanonisoi)
4. **Soveltaa** `classify_baird_taylor`-luokitusta
5. **Validoi** mole/weight-summat, SMILES, säilytysolot

## Periaate: LLM ei päätä luokituksia

Älä **koskaan** editoi `gfa_class`- tai `label_confidence`-kenttiä käsin. Jos rivi näyttää väärin luokitellulta, korjaa sen syöttötiedot (esim. `induction_time_censored`, `storage_T_C`, `experimental_protocol`) ja anna luokituksen päivittyä automaattisesti, kun ajat notebookin uudelleen.

Sama koskee SMILES-merkkijonoja: ne tulevat PubChem:istä RDKit-kanonisoinnin kautta, ei LLM:n muistista. Jos PubChem ei tunnista nimeä, rivi merkitään `needs_review=True` ja korjataan käsin `*_raw.json`-tiedostoon.

## Solu 2 — Imports ja konfiguraatio

In [21]:
import json
from pathlib import Path

import pandas as pd

from coamorphous.corpus.schema import build_schema, load_schema_yaml
from coamorphous.extraction.enrich import (
    assign_pair_id,
    raw_pair_to_master_row,
)
from coamorphous.extraction.extraction_schema import RawPair
from coamorphous.extraction.validate import run_all_validations

# --- Säädettävät parametrit per paperi ---------------------------------------
# Korvaa nämä jokaisen lähteen kohdalla.
PDF_PATH = Path("data/raw/lobmann_2012.pdf")
SOURCE_DOI = "10.1016/j.ijpharm.2012.05.016"
SOURCE_FIRST_AUTHOR = "lobmann"   # esim. "fink"
SOURCE_YEAR = 2012
EXTRACTED_BY = "claude_code"

# Polut, jotka johdetaan automaattisesti yllä olevista parametreista.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_JSON_PATH = PROJECT_ROOT / "data" / "interim" / f"{SOURCE_FIRST_AUTHOR}_{SOURCE_YEAR}_raw.json"
INTERIM_CSV_PATH = PROJECT_ROOT / "data" / "interim" / f"{SOURCE_FIRST_AUTHOR}_{SOURCE_YEAR}.csv"
SCHEMA_YAML = PROJECT_ROOT / "configs" / "corpus_schema.yaml"

print(f"PDF: {PDF_PATH}")
print(f"Raw JSON output: {RAW_JSON_PATH}")
print(f"Interim CSV output: {INTERIM_CSV_PATH}")

PDF: data\raw\lobmann_2012.pdf
Raw JSON output: c:\Users\perhe\vscode\coamorphous\data\interim\lobmann_2012_raw.json
Interim CSV output: c:\Users\perhe\vscode\coamorphous\data\interim\lobmann_2012.csv


## Vaihe 1 — LLM-raakaekstraktio

Avaa PDF VS Codessa rinnakkain ja pyydä Claude Codea (sivupaneeli) ekstraktoimaan parit alla olevalla promptilla. Tallenna tulos polkuun `RAW_JSON_PATH`.

### Prompt Claude Codelle (kopioi sivupaneeliin)

```
Lue PDF: {PDF_PATH}

Ekstraktoi KAIKKI ko-amorfiset parit RawPair-skeeman mukaisesti
(src/coamorphous/extraction/extraction_schema.py).

KRIITTISET SÄÄNNÖT:
1. Älä päättele tai laske kanonisia SMILES:eja, InChIKeyjä, gfa_class:ia,
   label_confidence:ia tai mitään pari-tason deskriptoreita - ne lasketaan
   automaattisesti.
2. Anna ratio_source_quote eksakti lainaus lähteestä (esim. "1:1 mol"
   tai "70:30 w/w").
3. Jos suhteita on annettu massoina, anna molemmat (weight_fraction ja
   mole_fraction laskettuna massoista, KUNNES Tm:t ovat tiedossa).
4. Aseta induction_time_censored TARKASTI:
   - True = kokeilu loppui ilman havaittua kiteytymistä
   - False = kiteytyminen havaittiin annetulla ajanhetkellä
5. Aseta experimental_protocol parhaalla mahdollisella tarkkuudella:
   - 40 °C / 75 % RH -> ich_q1a_accelerated
   - 25 °C / 60 % RH -> ich_q1a_long_term
   - Kuiva (P2O5/silica), <30 vrk -> dry_short_term
   - Tg+15 K -mittaus -> tg_plus_15K
   - Pelkkä DSC-uusinta -> dsc_in_situ
   - Muu / epäselvä -> non_standard
6. Aseta needs_review=True ja review_reasons-listalle, jos:
   - Lähde on epäjohdonmukainen (esim. taulukon otsikko vs. tekstin numerot)
   - Sensurointi on epäselvä
   - Protokolla ei sovi mihinkään selvästi
   - Stabiiliuskoetta ei raportoitu lainkaan (vain DSC tai PXRD)
7. source_quote on 1-2 lauseen lainaus lähteestä, joka tukee tämän
   parin tietoja - tärkeää auditointiin myöhemmin.

Palauta JSON-listana, jossa jokainen alkio on RawPair-skeeman mukainen dict.
Tallenna tiedostoon: data/interim/{first_author}_{year}_raw.json
```

Kun JSON on tallennettu, jatka alla.

In [22]:
with open(RAW_JSON_PATH, encoding="utf-8") as f:
    raw_data = json.load(f)

# Pydantic-validointi: kentät, tyypit, enumit, mole_fraction-rajat.
raw_pairs = [RawPair(**d) for d in raw_data]
print(f"Validoitu {len(raw_pairs)} paria.")

# Listaa heti needs_review-merkityt rivit, jotta ne näkyvät ennen rikastusta.
needs_review_now = [(i, p) for i, p in enumerate(raw_pairs) if p.needs_review]
if needs_review_now:
    print(f"\n[!] {len(needs_review_now)} paria merkitty needs_review jo ekstraktiossa:")
    for i, p in needs_review_now:
        print(f"  [{i}] {p.drug_A_name_raw}-{p.drug_B_name_raw}: {p.review_reasons}")

Validoitu 15 paria.

[!] 10 paria merkitty needs_review jo ekstraktiossa:
  [0] Simvastatin-Glipizide: ['Protokolla 4 °C/0% RH (P2O5-desikkaattori) ei sovi mihinkään standardiluokkaan: kuiva mutta seuranta-aika >30 vrk, ei dry_short_term']
  [1] Simvastatin-Glipizide: ['Protokolla 25 °C/0% RH (P2O5-desikkaattori) ei sovi standardiin: ei ICH (RH=0%) eikä dry_short_term (>30 vrk)', 'FTIR ja XRPD erivät merkittävästi (14 vrk vs 44 vrk); kerätty XRPD-arvo voi olla aliarvio']
  [3] Simvastatin-Glipizide: ['Protokolla 4 °C/0% RH (P2O5-desikkaattori) ei sovi mihinkään standardiluokkaan']
  [4] Simvastatin-Glipizide: ['Protokolla 25 °C/0% RH (P2O5) ei sovi standardiin (RH=0%, >30 vrk seuranta)', 'FTIR ja XRPD erivät (35 vrk vs 52 vrk)']
  [6] Simvastatin-Glipizide: ['Protokolla 4 °C/0% RH (P2O5) ei sovi mihinkään standardiluokkaan']
  [7] Simvastatin-Glipizide: ['Protokolla 25 °C/0% RH (P2O5) ei sovi standardiin (RH=0%, >30 vrk seuranta)']
  [9] Simvastatin-Glipizide: ['Protokolla 4 °C/0% RH (

## Vaihe 2 — Rikasta + luokittele

`raw_pair_to_master_row` tekee per pari:

1. Hakee PubChem:istä SMILES + InChIKey + CID molemmille lääkkeille
2. RDKit-kanonisoinnin
3. `classify_baird_taylor`-luokituksen `experimental_protocol`-tietoinen
4. Yhdistää lähdemetadatan ja palauttaa kaikki 54 master-CSV:n saraketta

PubChem-haku ei ole instant — varaa muutama sekunti per pari.

In [23]:
source_metadata = {
    "source_doi": SOURCE_DOI,
    "source_first_author": SOURCE_FIRST_AUTHOR,
    "source_year": SOURCE_YEAR,
}

rows = [
    raw_pair_to_master_row(p, source_metadata, EXTRACTED_BY)
    for p in raw_pairs
]
rows = assign_pair_id(rows, SOURCE_FIRST_AUTHOR, SOURCE_YEAR)

df = pd.DataFrame(rows)
print(f"Rikastettu {len(df)} riviä, {len(df.columns)} saraketta.")
print("\nLuokat (gfa_class x label_confidence):")
print(df.groupby(["gfa_class", "label_confidence"], dropna=False).size())

Rikastettu 15 riviä, 54 saraketta.

Luokat (gfa_class x label_confidence):
gfa_class  label_confidence
2          high                 5
           low                 10
dtype: int64


## Vaihe 3 — Validoi

Kahdenlaisia tarkistuksia:

* **Riviä-kohti** (`run_all_validations`): mole/weight-summat, SMILES-validius, säilytysolojen yhteensopivuus protokollan kanssa.
* **Skeematason** (Pandera, `build_schema`): tyypit, enumit, vaaditut sarakkeet, sarakejärjestys.

Jos jompikumpi epäonnistuu, korjaa lähdön JSON ja aja Vaihe 2 uudelleen.

In [24]:
validation_errors = []
for idx, row in df.iterrows():
    errors = run_all_validations(row.to_dict())
    if errors:
        validation_errors.append((idx, row["pair_id"], errors))

if validation_errors:
    print(f"[!] {len(validation_errors)} riviä ei läpäissyt riviä-kohti -validointia:")
    for idx, pid, errs in validation_errors:
        print(f"  [{idx}] {pid}:")
        for e in errs:
            print(f"    - {e}")
else:
    print("[OK] Kaikki rivit läpäisivät riviä-kohti -validoinnin.")

# Pandera-skeemavalidointi yhdistetylle DataFramelle.
schema = build_schema(SCHEMA_YAML)
try:
    schema.validate(df, lazy=True)
    print("[OK] Pandera-skeema validoitu.")
except Exception as e:
    print(f"[!] Pandera-virhe:\n{e}")

[OK] Kaikki rivit läpäisivät riviä-kohti -validoinnin.
[OK] Pandera-skeema validoitu.


## Vaihe 4 — Tarkista needs_review / low_confidence -rivit

Listataan rivit, jotka vaativat ihmissilmää: joko LLM merkitsi `needs_review=True` ekstraktiossa, tai luokitin antoi `label_confidence='low'/'borderline'`.

In [25]:
# Master-CSV ei sisällä needs_review-saraketta (se on RawPair-tason kenttä),
# joten käytämme label_confidence- ja notes-kenttiä proxynä.
review_mask = df["label_confidence"].isin(["low", "borderline"]) | df["notes"].fillna("").str.contains(r"\[review\]", regex=True)
review_df = df[review_mask]

if len(review_df) > 0:
    print(f"Tarkista {len(review_df)} riviä manuaalisesti:")
    display_cols = [
        "pair_id", "drug_A_name", "drug_B_name",
        "gfa_class", "label_confidence", "experimental_protocol", "notes",
    ]
    print(review_df[display_cols].to_string())
else:
    print("[OK] Ei review-merkittyjä eikä low/borderline-rivejä.")

Tarkista 10 riviä manuaalisesti:
           pair_id  drug_A_name drug_B_name  gfa_class label_confidence experimental_protocol                                                                                                                                                                                                                                                                                                                                                                                      notes
0   lobmann2012_01  Simvastatin   Glipizide          2              low          non_standard                Storage at 4 °C/0% RH (P2O5) is non-standard ICH protocol; XRPD detection at day 128, FTIR at day 133. Authors found no molecular SVS-GPZ interactions; stability attributed to GPZ acting as anti-plasticizer (higher Tg). | [review] Protokolla 4 °C/0% RH (P2O5-desikkaattori) ei sovi mihinkään standardiluokkaan: kuiva mutta seuranta-aika >30 vrk, ei dry_short_term
1   lobmann2012_02  S

## Vaihe 5 — Tallenna interim-CSV

In [26]:
df.to_csv(INTERIM_CSV_PATH, index=False)
print(f"[OK] Tallennettu: {INTERIM_CSV_PATH} ({len(df)} riviä)")
print("\nSEURAAVAT TOIMET:")
print("1. Tarkista yllä listatut needs_review / low-confidence -rivit käsin.")
print(f"2. Editoi tarvittaessa {RAW_JSON_PATH.name}, aja Vaihe 2-5 uudelleen.")
print("3. Kun OK, lisää lähde merge.py:n yhdistämislistaan ja päivitä master-CSV.")

[OK] Tallennettu: c:\Users\perhe\vscode\coamorphous\data\interim\lobmann_2012.csv (15 riviä)

SEURAAVAT TOIMET:
1. Tarkista yllä listatut needs_review / low-confidence -rivit käsin.
2. Editoi tarvittaessa lobmann_2012_raw.json, aja Vaihe 2-5 uudelleen.
3. Kun OK, lisää lähde merge.py:n yhdistämislistaan ja päivitä master-CSV.
